In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

data = torch.tensor([
    [ 1.0, 1.0,  9.0],
    [ 6.0, 2.0, 36.0],
    [ 3.0, 4.0, 25.0],
    [ 9.0, 1.0, 40.0],
    [ 2.0, 3.0, 18.0],
    [12.0, 2.0, 48.0],
    [ 4.0, 1.0, 24.0],
    [15.0, 4.0, 58.0],
    [ 5.0, 3.0, 33.0],
    [18.0, 2.0, 60.0],
    [ 7.0, 4.0, 42.0],
    [20.0, 1.0, 62.0],
    [ 2.5, 2.0, 18.5],
    [ 8.0, 3.0, 42.0],
    [13.0, 1.0, 48.0],
    [17.0, 4.0, 62.0]
], dtype=torch.float32)

train_X = data[0:12,0:2]
train_y = data[0:12,2:3]
val_X = data[12:16,0:2]
val_y = data[12:16,2:3]

print(train_X.shape, train_y.shape)
print(val_X.shape, val_y.shape)

# 只使用训练集计算统计量，统一沿 dim=0 计算
X_mean = train_X.mean(dim=0)
X_std = train_X.std(dim=0)
y_mean = train_y.mean(dim=0)
y_std = train_y.std(dim=0)

# 标准化训练集
train_X_norm =  (train_X - X_mean) / X_std
train_y_norm = (train_y - y_mean) / y_std

# 验证集沿用上面同一套统计量
val_X_norm = (val_X - X_mean) / X_std
val_y_norm = (val_y - y_mean) / y_std

torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(2,100),
    nn.ReLU(),
    nn.Linear(100,1)
)
loss_function = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(3000):
    optimizer.zero_grad()
    outputs = model(train_X_norm)
    Loss = loss_function(outputs, train_y_norm)
    Loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        print(f'Epoch [{epoch+1}/3000], Loss: {Loss.item():.4f}')

with torch.no_grad():
    # 分别预测训练集、验证集
    train_pred_norm = model(train_X_norm)
    val_pred_norm = model(val_X_norm)

    # 将预测还原为分钟
    train_pred_minutes = train_pred_norm*y_std +y_mean
    val_pred_minutes = val_pred_norm *y_std + y_mean

    # 与原始分钟数标签比较，计算 MSE
    train_mse = nn.MSELoss()(train_pred_minutes, train_y)
    val_mse = nn.MSELoss()(val_pred_minutes, val_y)

print("训练 MSE：", train_mse.item())
print("验证 MSE：", val_mse.item())





torch.Size([12, 2]) torch.Size([12, 1])
torch.Size([4, 2]) torch.Size([4, 1])
Epoch [500/3000], Loss: 0.0149
Epoch [1000/3000], Loss: 0.0079
Epoch [1500/3000], Loss: 0.0047
Epoch [2000/3000], Loss: 0.0028
Epoch [2500/3000], Loss: 0.0017
Epoch [3000/3000], Loss: 0.0010
训练 MSE： 0.2934456765651703
验证 MSE： 0.8322902321815491
